# LTO experiment 2 -- Qwen3-1.7B on CommonsenseQA

Classifier trained on metrics only (`USE_HIDDEN=False`) -- isolates whether the 4 representation-geometry metrics alone carry the correctness signal, without raw hidden states. Includes the extended `lto.analysis` diagnostics (flip counts, qualitative inspection, reward-gap/confidence buckets, and a feature-engineering + grouped-CV classifier sweep) on top of the shared pipeline used in notebook 1.

In [ ]:
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes datasets nltk pandas tqdm
!pip install -q "git+https://github.com/Braj406/Input-Transformation-LTO.git"


In [ ]:
from lto import ensure_nltk_data, set_seed, SEED, DEVICE, DTYPE, GPU_NAME, MODELS, COMPACT_CFG, run_multi
from lto import analysis, model_io
from lto.classifier import make_loader, sweep_learning_rates
from lto.lto_algorithm import group_tasks_by_question, run_lto_algorithm1, lto_stability_across_seeds
from lto.data import load_examples
import torch, transformers, platform

ensure_nltk_data()
set_seed(SEED)
print(f"device={DEVICE} dtype={DTYPE} gpu={GPU_NAME}")
print(f"torch={torch.__version__} transformers={transformers.__version__} python={platform.python_version()}")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive/LTO"


In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')   # Colab secret: click the key icon in the left sidebar, add HF_TOKEN
    login(token=hf_token)
    print("Logged in via Colab secret.")
except Exception:
    print("No HF_TOKEN secret found -- falling back to interactive login (paste your token when prompted).")
    login()


## Experiment config

In [ ]:
MODEL_KEY = "qwen3-1.7b"
DATASET   = "commonsense_qa"
OUT_NAME  = "detv1"
TAG       = f"{MODELS[MODEL_KEY]['short']}__{DATASET}" + (f"__{OUT_NAME}" if OUT_NAME else "")

print("Model:", MODELS[MODEL_KEY]["id"])
print("Inference: greedy, temperature=0 (deterministic). Temperature is used ONLY for rewrites.")


## Collect trajectories (baseline + transformed variants)

In [ ]:
TRANSFORMS = [
    {"type": "llm_knowledge",   "k": 5, "attempts": 3, "n_exemplars": 4},
    {"type": "llm_simplify",    "k": 1, "attempts": 2},
    {"type": "llm_simplify_v2", "k": 1, "attempts": 2},
    {"type": "llm_simplify_v3", "k": 1, "attempts": 2},
]

recs, path = run_multi(
    model_key=MODEL_KEY, dataset_name=DATASET, transforms=TRANSFORMS,
    n_examples=300, drive_dir=DRIVE_DIR, out_name=OUT_NAME, resume=True, n_pool=64, seed=SEED,
)


## Load records, build the row-level dataset, split by question

In [ ]:
import os
import torch

PT_PATH = os.path.join(DRIVE_DIR, TAG + ".pt")
recs = torch.load(PT_PATH, weights_only=False)

CLASSIFIER_KEYS = ["llm_knowledge_g", "llm_simplify_g", "llm_simplify_v2_g", "llm_simplify_v3_g"]

analysis.usable_row_report(recs, CLASSIFIER_KEYS)


In [ ]:
rows = analysis.collect_rows(recs, CLASSIFIER_KEYS, include_original=True)
print(f"Total usable rows for training: {len(rows)}")

L = rows[0]["metrics"].shape[0]
RAW_HIDDEN_DIM = rows[0]["hidden"].shape[1]
N_METRICS = rows[0]["metrics"].shape[1]
print(f"layers={L}  raw_hidden_dim={RAW_HIDDEN_DIM}  n_metrics={N_METRICS}")

idxs, labels, train_i, val_i, test_i = analysis.grouped_split(rows, seed=SEED)

metrics_stack = analysis.standardize_metrics(rows, train_i)

USE_PCA = False   # set True to compress hidden states before feeding the transformer
PCA_DIM = 24
hidden_stack, pca = analysis.maybe_pca_hidden(rows, train_i, use_pca=USE_PCA, pca_dim=PCA_DIM, seed=SEED)

HIDDEN_DIM = hidden_stack.shape[-1]
label_stack = torch.tensor(labels, dtype=torch.float32)
print(f"final hidden feature dim per layer-token: {HIDDEN_DIM}")


## Train the trajectory classifier (reward model)

Start with `USE_HIDDEN=False` to isolate whether the 4 metrics alone carry the signal; flip to `True` (with `USE_PCA=True` in the split cell above) once metrics-only is confirmed.

In [ ]:
USE_HIDDEN = False

BATCH_SIZE = 32
train_loader = make_loader(hidden_stack, metrics_stack, label_stack, train_i, BATCH_SIZE, shuffle=True)
val_loader   = make_loader(hidden_stack, metrics_stack, label_stack, val_i,   BATCH_SIZE, shuffle=False)
test_loader  = make_loader(hidden_stack, metrics_stack, label_stack, test_i,  BATCH_SIZE, shuffle=False)

MODEL_KWARGS = dict(n_layers_seq=L, hidden_dim=HIDDEN_DIM, n_metrics=N_METRICS,
                    d_model=32, nhead=2, dim_feedforward=64, dropout=0.5, use_hidden=USE_HIDDEN)


In [ ]:
LR_GRID = [1e-4, 3e-4, 1e-3]
N_RUNS_PER_LR = 10
EPOCHS, PATIENCE = 200, 25

model, all_results, best_lr = sweep_learning_rates(
    MODEL_KWARGS, train_loader, val_loader, label_stack, train_i,
    lr_grid=LR_GRID, n_runs_per_lr=N_RUNS_PER_LR, base_seed=SEED,
    device=DEVICE, epochs=EPOCHS, patience=PATIENCE,
)


## LTO Algorithm 1: rejection-sampling selection on the held-out test tasks

In [ ]:
test_tasks_grouped = group_tasks_by_question(idxs, test_i)

n_per_task = [len(v) for v in test_tasks_grouped.values()]
print(f"LTO evaluating {len(test_tasks_grouped)} held-out tasks "
      f"({sum(n_per_task)} trajectories)")
print(f"candidates/task -> mean={sum(n_per_task)/len(n_per_task):.1f} "
      f"min={min(n_per_task)} max={max(n_per_task)}")

print("\nEXECUTING LATENT THINKING OPTIMIZATION (ALGORITHM 1)")
lto_result = run_lto_algorithm1(test_tasks_grouped, hidden_stack, metrics_stack, label_stack,
                                model, DEVICE, beta=0.05, seed=0)
greedy_acc = lto_result["greedy_acc"]

lto_accs = lto_stability_across_seeds(test_tasks_grouped, hidden_stack, metrics_stack, label_stack,
                                      model, DEVICE, beta=0.05, n_seeds=10)
print(f"(Greedy for comparison: {greedy_acc:.2f}%)")


## Flip-count transitions: original vs. classifier-selected candidate

In [ ]:
orig_correct_by_idx = {r["idx"]: bool(r["original"]["correct"]) for r in recs}

greedy_correct_by_idx = analysis.greedy_correct_by_task(model, hidden_stack, metrics_stack, label_stack,
                                                        test_tasks_grouped, DEVICE)
greedy_trans = analysis.compute_transitions(greedy_correct_by_idx, orig_correct_by_idx)
print("GREEDY vs ORIGINAL (deterministic):")
print(f"  incorrect -> correct : {greedy_trans['incorrect_to_correct']}")
print(f"  correct -> incorrect : {greedy_trans['correct_to_incorrect']}")
print(f"  stayed correct       : {greedy_trans['stayed_correct']}")
print(f"  stayed incorrect     : {greedy_trans['stayed_incorrect']}")
print(f"  net change           : "
      f"{greedy_trans['incorrect_to_correct'] - greedy_trans['correct_to_incorrect']:+d} tasks")

b, c, n, p = analysis.mcnemar_on_transitions(greedy_trans)
print(f"  McNemar test: b={b} c={c} n={n}  p={p:.3f}")

lto_trans_runs = analysis.lto_transitions_across_seeds(model, hidden_stack, metrics_stack, label_stack,
                                                       test_tasks_grouped, orig_correct_by_idx, DEVICE,
                                                       beta=0.05, n_seeds=10)


## Qualitative inspection of correct<->incorrect flips

In [ ]:
examples_ds, _ = load_examples(DATASET, len(recs))
recs_by_idx = {r["idx"]: r for r in recs}

c2o_tasks = [idx for idx in test_tasks_grouped
             if orig_correct_by_idx[idx] and not greedy_correct_by_idx[idx]]
o2c_tasks = [idx for idx in test_tasks_grouped
             if not orig_correct_by_idx[idx] and greedy_correct_by_idx[idx]]

print(f"correct -> incorrect tasks: {c2o_tasks}")
print(f"incorrect -> correct tasks: {o2c_tasks}")

print("\n# CORRECT -> INCORRECT (the ones to investigate)")
for idx in c2o_tasks:
    analysis.inspect_task(idx, model, recs_by_idx, examples_ds, CLASSIFIER_KEYS, DEVICE)

print("\n# INCORRECT -> CORRECT (for contrast)")
for idx in o2c_tasks:
    analysis.inspect_task(idx, model, recs_by_idx, examples_ds, CLASSIFIER_KEYS, DEVICE)


## Reward compression: does the model actually discriminate between candidates?

In [ ]:
all_probs, gaps = analysis.reward_gap_stats(model, hidden_stack, metrics_stack, test_tasks_grouped, DEVICE)

bucket_results = analysis.confidence_bucket_stats(model, hidden_stack, metrics_stack, label_stack,
                                                   test_tasks_grouped, orig_correct_by_idx, DEVICE,
                                                   gap_threshold=0.01)


## Feature engineering + honest grouped-CV classifier sweep

Test questions are held out before any sweeping; CV runs only on dev questions. Compares metrics-only / metrics+delta / +gen-length / +source features across logistic regression, random forest, and hist-gradient-boosting classifiers.

In [ ]:
_ = analysis.run_feature_sweep(recs, CLASSIFIER_KEYS, seed=SEED, test_size=0.20)
